# RECELL-AI — Training Classification (YOLOv8-cls)

Klasifikasi kondisi baterai: **KOSONG / SEHAT / KARAT / SOBEK** (single-label).

Alur: Roboflow (dataset) -> Colab (train, GPU gratis) -> unduh `best.pt` -> export TensorRT di Jetson.

**Runtime > Change runtime type > GPU** sebelum mulai.

**API key — sekali saja, tak usah ketik tiap run:** klik ikon kunci (Secrets) di sidebar kiri, tambah `ROBOFLOW_API_KEY` = private key-mu, aktifkan akses notebook. Selesai. (Tak ditanam di file -> tak ke-commit ke GitHub.)

In [ ]:
!pip install -q roboflow ultralytics

In [ ]:
# Ambil API key dari Colab Secrets (set sekali, tak pernah ketik lagi). Fallback: ketik manual.
import os
from roboflow import Roboflow
try:
    from google.colab import userdata
    api_key = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    import getpass
    api_key = os.environ.get("ROBOFLOW_API_KEY") or getpass.getpass("Roboflow Private API Key: ")

WORKSPACE = "legacy-gyzej"
PROJECT   = "baterai-legacy"
VERSION   = 1   # ganti sesuai versi yang kamu Generate di Roboflow

rf = Roboflow(api_key=api_key)
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("folder")
print("dataset di:", dataset.location)

In [ ]:
# ultralytics-cls minta subfolder 'train' & 'val'; Roboflow kasih 'valid'. Rename.
import os
root = dataset.location
if os.path.isdir(f"{root}/valid") and not os.path.isdir(f"{root}/val"):
    os.rename(f"{root}/valid", f"{root}/val")
print(sorted(os.listdir(root)))

In [ ]:
from ultralytics import YOLO
# nano-cls: ringan & cepat di Jetson setelah export TensorRT. imgsz 224 standar cls.
model = YOLO("yolov8n-cls.pt")
model.train(data=root, epochs=100, imgsz=224, batch=64, project="recell", name="cls")

In [ ]:
# Unduh best.pt
from google.colab import files
best = "recell/cls/weights/best.pt"
print("best:", best)
files.download(best)

## Deploy ke Jetson
1. Salin `best.pt` -> `jetson/models/weights/best_cls.pt` di Jetson.
2. Export TensorRT (di Jetson):
   ```bash
   yolo export model=jetson/models/weights/best_cls.pt format=engine half=True imgsz=224
   ```
3. Arahkan pipeline kamera ke engine cls + ganti `process_ai_results` ke top-1 kelas (gate KOSONG). Minta Claude lanjutkan bagian ini setelah model jadi.